# Transformer encoders and a fused multimodal transformer

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ashford-A/UniVI/blob/main/docs/tutorials/experimental/transformer_encoders.ipynb)

> **Experimental.** Transformer encoders were not used for the results in the paper. This notebook
> compares them with the standard MLP encoders on identical inputs and reports whatever comes out; it
> does not assume that attention helps.

UniVI can encode a modality with a transformer over **tokens**, where each token is one of the cell's
most prominent features (a highly expressed gene, an accessible peak) carrying its value, its rank,
and a learned embedding of *which* feature it is. It can also encode all modalities jointly with one
**fused transformer** over the concatenated tokens, so that gene tokens and peak tokens attend to each
other before the latent posterior is formed.

On 10x Multiome PBMCs this notebook:

1. builds token-friendly inputs (z-scored RNA, TF-IDF-weighted peaks);
2. trains four models on the same cells and features: MLP (paper objective, `v1`), per-modality
   transformers (`v1`), MLP (`v2`), and MLP experts plus a fused transformer (`v2`);
3. compares alignment, label transfer, ATAC → RNA prediction, fused-embedding quality and training time;
4. opens up the fused transformer: how much attention crosses modalities in each layer, which gene–peak
   token pairs it links, and whether a gene token attends more to **cis** peaks (near its TSS) than to
   other peaks present in the same cell, after removing gene- and peak-level attention effects.

The last test is a falsifiable check: attention weights are not explanations, but if the fused encoder
had learned nothing about genomic proximity, cis peaks would receive no more of a gene's attention than
other peaks.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "univi[tutorials]>=1.1" "pandas==2.2.3"

In [ ]:
import time
from dataclasses import dataclass

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import torch
from IPython.display import display
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier

import univi
import univi.datasets as uds
from univi import ModalityConfig, TrainingConfig, UniVIConfig, UniVIMultiModalVAE, UniVITrainer
from univi.config import TokenizerConfig, TransformerConfig
from univi.evaluation import (cross_modal_predict, encode_adata, encode_fused_adata_pair, evaluate_alignment,
                              pearson_corr_per_feature)
from univi.interpretability import fused_encode_with_meta_and_attn, top_cross_modal_feature_pairs_from_attn
from univi.preprocessing import RNAPreprocessor, split_by_label
from univi.utils.seed import set_seed
from univi.workflows import make_loader

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
set_seed(0)
dense = lambda x: x.toarray() if sp.issparse(x) else np.asarray(x)
print(f"UniVI {univi.__version__} on {device}")

In [ ]:
N_EPOCHS = 300
BATCH_SIZE = 256
N_HVG = 2000
N_PEAKS = 20000            # most frequently open peaks in training cells
ATAC_RECON_WEIGHT = 0.2    # balances the ATAC reconstruction sum against RNA's; tune if needed
N_TOKENS_RNA = 128         # genes per cell given to the transformer (the cell's highest z-scores)
N_TOKENS_ATAC = 128        # peaks per cell (the cell's highest TF-IDF values)
D_TOKEN = 64               # token embedding size before the transformer
TRANSFORMER = dict(d_model=128, num_heads=4, num_layers=2, dim_feedforward=256, dropout=0.1, attn_dropout=0.1)
CIS_WINDOW_BP = 100_000    # gene-peak pairs closer than this (same chromosome) count as cis
N_PERMUTATIONS = 200       # peak-coordinate shuffles for the cis-attention null
MAX_ATTENTION_CELLS = 2000

## 1. Data and token-friendly inputs

A top-k tokenizer keeps each cell's **largest** values. For RNA we z-score log-normalized expression,
so the kept genes are those most above their average level in that cell. For ATAC, binarized peaks
would tie (every open peak equals 1), so we use Signac-style TF-IDF, `log1p(TF × IDF × 1e4)` with the
IDF fit on training cells: the kept peaks are the cell's open peaks that are rarest across cells,
which is where cell identity lives. All four models receive exactly these inputs.

In [ ]:
data = uds.pbmc_multiome_10k()
rna, atac = data["rna"], data["atac"]
splits = split_by_label(rna.obs["cell_type"], train_fraction=0.8, val_fraction=0.1, seed=0)
print({k: len(v) for k, v in splits.items()})

rna_prep = RNAPreprocessor(n_hvg=N_HVG, scale=True).fit(rna[splits["train"]])
train_counts = sp.csr_matrix(atac[splits["train"]].layers["counts"])
freq = np.asarray((train_counts > 0).mean(axis=0)).ravel()
keep = np.argsort(-freq, kind="stable")[: min(N_PEAKS, int((freq > 0).sum()))]
df = np.asarray((train_counts[:, keep] > 0).sum(axis=0)).ravel()
idf = np.log1p(train_counts.shape[0] / (1.0 + df)).astype(np.float32)


def tfidf(adata_counts):
    x = sp.csr_matrix(adata_counts.layers["counts"][:, keep], dtype=np.float32)
    x.data = np.ones_like(x.data)                                  # binarize, then TF-IDF
    tf = sp.diags(1.0 / np.maximum(np.asarray(x.sum(axis=1)).ravel(), 1.0)) @ x
    out = sp.csr_matrix(tf.multiply(idf[None, :]))
    out.data = np.log1p(out.data * 1e4).astype(np.float32)
    return ad.AnnData(out, obs=adata_counts.obs.copy(), var=atac.var.iloc[keep].copy())


parts = {k: {"rna": rna_prep.transform(rna[i]), "atac": tfidf(atac[i])} for k, i in splits.items()}
train, val, test = parts["train"], parts["val"], parts["test"]
print("RNA", train["rna"].shape, "| ATAC", train["atac"].shape)

## 2. Four models on the same inputs

A top-k token says "this feature has this value and this rank in this cell". Without an identity
embedding the transformer would not know *which* gene or peak a token is, so the tokenizers below
switch on a learned **feature-ID embedding**. In UniVI 1.1 the tokenizer reads this option
(`use_feature_embedding`, `feature_emb_dim`, `feature_emb_mode`) with `getattr`, but `TokenizerConfig`
does not declare it as a field. Setting it as a plain attribute is not enough: the fused encoder copies
each tokenizer config with `dataclasses.replace`, which keeps only fields, so the option would silently
disappear there. A small dataclass subclass that declares the three options as fields works in both
the per-modality and the fused encoders. (The reference bundle written by `save_reference` cannot be
read back by `load_reference` with this subclass; Section 5 shows how to save and restore instead.)

The fused transformer is only trained by the `v2` objective (in `v1` its posterior never enters the
reconstruction loss), so an MLP model with `v2` is included to separate the effect of the fused encoder
from the effect of the objective. `recon_normalize_by_dim=False` keeps `v2` reconstruction sums on the
same scale as `v1`, so the same `beta` and `gamma` are used throughout.

In [ ]:
@dataclass
class IdTokenizerConfig(TokenizerConfig):
    """TokenizerConfig plus the feature-ID embedding options the UniVI 1.1 tokenizer reads."""
    use_feature_embedding: bool = True
    feature_emb_dim: int = 64
    feature_emb_mode: str = "add"


def tokenizer(n_features, n_tokens):
    return IdTokenizerConfig(mode="topk_channels", n_tokens=n_tokens, channels=("value", "rank"),
                             n_features=int(n_features), feature_emb_dim=D_TOKEN)


def modalities(encoder):
    kw = lambda n, k: dict(encoder_type="transformer", transformer=TransformerConfig(**TRANSFORMER),
                           tokenizer=tokenizer(n, k)) if encoder == "transformer" else \
        dict(tokenizer=tokenizer(n, k))           # MLP expert; the tokenizer is used only by a fused encoder
    n_rna, n_atac = train["rna"].n_vars, train["atac"].n_vars
    return [ModalityConfig("rna", n_rna, [512, 256, 128], [128, 256, 512], likelihood="gaussian",
                           **kw(n_rna, N_TOKENS_RNA)),
            ModalityConfig("atac", n_atac, [512, 256, 128], [128, 256, 512], likelihood="gaussian",
                           recon_weight=ATAC_RECON_WEIGHT, **kw(n_atac, N_TOKENS_ATAC))]


common = dict(latent_dim=30, beta=1.25, gamma=4.35, encoder_dropout=0.10, decoder_dropout=0.05,
              kl_anneal_start=50, kl_anneal_end=85, align_anneal_start=75, align_anneal_end=110)
specs = {
    "MLP (v1)": (UniVIConfig(modalities=modalities("mlp"), **common), dict(loss_mode="v1", v1_recon="avg")),
    "transformer (v1)": (UniVIConfig(modalities=modalities("transformer"), **common),
                         dict(loss_mode="v1", v1_recon="avg")),
    "MLP (v2)": (UniVIConfig(modalities=modalities("mlp"), **common),
                 dict(loss_mode="v2", recon_normalize_by_dim=False)),
    "fused transformer (v2)": (UniVIConfig(modalities=modalities("mlp"), fused_encoder_type="multimodal_transformer",
                                           fused_transformer=TransformerConfig(**TRANSFORMER), **common),
                               dict(loss_mode="v2", recon_normalize_by_dim=False)),
}

models, histories, minutes = {}, {}, {}
for name, (cfg, kwargs) in specs.items():
    set_seed(0)
    model = UniVIMultiModalVAE(cfg, normalize_v1_terms=True, **kwargs)
    n_params = sum(p.numel() for p in model.parameters())
    trainer = UniVITrainer(
        model, make_loader(train, batch_size=BATCH_SIZE, shuffle=True, drop_last=True),
        make_loader(val, batch_size=512),
        TrainingConfig(n_epochs=N_EPOCHS, batch_size=BATCH_SIZE, lr=1e-3, weight_decay=1e-4, device=device,
                       early_stopping=True, patience=50, best_epoch_warmup=110, log_every=100, grad_clip=5.0))
    started = time.time()
    histories[name] = trainer.fit()
    minutes[name] = (time.time() - started) / 60
    models[name] = model
    print(f"{name}: {n_params / 1e6:.1f}M parameters, best epoch {trainer.best_epoch}, {minutes[name]:.1f} min")

fused_tok = models["fused transformer (v2)"].fused_encoder.vec2tok
assert all(fused_tok[m].id_emb is not None for m in fused_tok), "feature-ID embeddings missing in fused encoder"

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 3.2))
for name, h in histories.items():
    ax.plot(h["val_loss"], label=name)
ax.set(xlabel="epoch", ylabel="validation loss", yscale="log", title="Validation loss (full beta, gamma)")
ax.legend(frameon=False, fontsize=8)
plt.show()

Losses of `v1` and `v2` models are different objectives, so compare curves only within an objective.

## 3. Benchmark on held-out cells

- **FOSCTTM** and **Recall@10**: are a cell's RNA and ATAC embeddings close to each other?
- **label transfer** (RNA → ATAC, k-NN) and its macro-F1 in the worse direction;
- **ATAC → RNA**: median per-gene Pearson correlation of RNA predicted from accessibility alone;
- **fused kNN accuracy**: 5-fold cross-validated 15-NN cell-type accuracy on the embedding that uses
  both modalities (the fused transformer's posterior, or the precision-weighted MoE for the others).

In [ ]:
labels = test["rna"].obs["cell_type"].astype(str).to_numpy()
observed = dense(test["rna"].X)
rows, fused_z = {}, {}
for name, model in models.items():
    z_r = encode_adata(model, test["rna"], modality="rna", device=device, latent="modality_mean")
    z_a = encode_adata(model, test["atac"], modality="atac", device=device, latent="modality_mean")
    m = evaluate_alignment(Z1=z_r, Z2=z_a, labels_source=labels, labels_target=labels, recall_ks=(10,))
    pred = cross_modal_predict(model, test["atac"], src_mod="atac", tgt_mod="rna", device=device)
    fz = encode_fused_adata_pair(model, {"rna": test["rna"], "atac": test["atac"]}, device=device,
                                 write_to_adatas=False)["mu"]
    fused_z[name] = fz
    folds = int(min(5, pd.Series(labels).value_counts().min())) if len(set(labels)) > 1 else 0
    acc = (cross_val_score(KNeighborsClassifier(15), fz, labels, cv=folds).mean() if folds >= 2 else np.nan)
    rows[name] = {"FOSCTTM ↓": m["foscttm_mean"], "Recall@10 ↑": m["recall_at_k"]["10"]["mean"],
                  "label transfer ↑": m["label_transfer_acc"], "macro-F1 (worse dir.) ↑": m["worst_direction_macro_f1"],
                  "ATAC→RNA median r ↑": float(np.nanmedian(pearson_corr_per_feature(observed, pred))),
                  "fused kNN acc ↑": acc, "train min": minutes[name]}
bench = pd.DataFrame(rows).T
bench.round(3)

This is a single seed on one dataset. Before concluding that one encoder is better, repeat the
comparison over a few seeds (differences between seeds can be as large as differences between models).

In [ ]:
fig, axes = plt.subplots(1, len(models), figsize=(3.6 * len(models), 3.4))
for ax, (name, fz) in zip(np.atleast_1d(axes), fused_z.items()):
    a = ad.AnnData(fz.astype(np.float32),
                   obs=pd.DataFrame({"cell_type": pd.Categorical(labels)}, index=test["rna"].obs_names))
    sc.pp.neighbors(a, n_neighbors=30, use_rep="X")
    sc.tl.umap(a, random_state=0)
    sc.pl.umap(a, color="cell_type", ax=ax, show=False, legend_loc="none", title=name, frameon=False)
plt.tight_layout()
plt.show()

## 4. Inside the fused transformer

`fused_encode_with_meta_and_attn` runs the fused encoder and returns the attention matrices of every
layer (averaged over heads) together with a token map: which slice of the sequence belongs to which
modality and, for top-k tokenizers, which feature each token was in each cell.

In [ ]:
fused = models["fused transformer (v2)"]
fused.eval()
n_cells = min(MAX_ATTENTION_CELLS, test["rna"].n_obs)
cells = np.random.default_rng(0).choice(test["rna"].n_obs, n_cells, replace=False)
X_rna, X_atac = dense(test["rna"].X[cells]), dense(test["atac"].X[cells])


def attention_batches(batch=256):
    for s in range(0, n_cells, batch):
        xb = {"rna": torch.as_tensor(X_rna[s:s + batch], dtype=torch.float32, device=device),
              "atac": torch.as_tensor(X_atac[s:s + batch], dtype=torch.float32, device=device)}
        _, _, tokmap, attn = fused_encode_with_meta_and_attn(fused, xb, return_attn=True)
        yield s, tokmap, attn


_, tokmap0, attn0 = next(attention_batches(8))
print("token slices:", tokmap0.slices, "| global CLS:", tokmap0.has_global_cls,
      "| layers:", len(attn0), "| attention shape:", tuple(attn0[0].shape))

### How much attention crosses modalities?

For each layer, the average attention a query token of one modality sends to key tokens of each
modality (rows sum to 1). Off-diagonal mass is cross-modal mixing.

In [ ]:
mods = list(tokmap0.slices)
mass = np.zeros((len(attn0), len(mods), len(mods)))
count = 0
for s, tokmap, attn in attention_batches():
    shift = 1 if tokmap.has_global_cls else 0
    for li, A in enumerate(attn):
        A = A.float().cpu().numpy()
        for i, qa in enumerate(mods):
            a0, a1 = [v + shift for v in tokmap.slices[qa]]
            for j, kb in enumerate(mods):
                b0, b1 = [v + shift for v in tokmap.slices[kb]]
                mass[li, i, j] += A[:, a0:a1, b0:b1].sum(-1).mean(-1).sum()
    count += A.shape[0]
mass /= count
fig, axes = plt.subplots(1, len(attn0), figsize=(2.8 * len(attn0), 2.6))
for li, ax in enumerate(np.atleast_1d(axes)):
    im = ax.imshow(mass[li], vmin=0, vmax=1, cmap="magma")
    for i in range(len(mods)):
        for j in range(len(mods)):
            ax.text(j, i, f"{mass[li, i, j]:.2f}", ha="center", va="center", color="w", fontsize=9)
    ax.set_xticks(range(len(mods)), [f"→{m}" for m in mods])
    ax.set_yticks(range(len(mods)), mods)
    ax.set_title(f"layer {li + 1}")
plt.tight_layout()
plt.show()

### Gene → peak pairs the fused encoder links most

`top_cross_modal_feature_pairs_from_attn` sums, over cells, the attention from gene tokens (queries)
to peak tokens (keys) for each cell's strongest pairs. Summed attention favors features that are tokens
in many cells, so read this as a list of frequent, strongly linked pairs. Distances to the gene's TSS
are added for orientation.

In [ ]:
def tss_table(var):
    need = ["chrom", "chromStart", "chromEnd", "strand"]
    if not set(need) <= set(var.columns):
        return None
    v = var[need].dropna()
    v = v[v["strand"].astype(str).isin(["+", "-"])]
    tss = np.where(v["strand"].astype(str) == "-", v["chromEnd"], v["chromStart"]).astype(np.int64)
    return pd.DataFrame({"chrom": v["chrom"].astype(str).to_numpy(), "tss": tss}, index=v.index)


tss = tss_table(rna.var)
pv = test["atac"].var
peak_chrom = pv["chrom"].astype(str).to_numpy() if "chrom" in pv else None
peak_mid = ((pv["chromStart"].astype(np.int64) + pv["chromEnd"].astype(np.int64)) // 2).to_numpy() \
    if {"chromStart", "chromEnd"} <= set(pv.columns) else None

_, tokmap_b, attn_b = next(attention_batches(512))
pairs = top_cross_modal_feature_pairs_from_attn(
    attn_b, tokmap_b, mod_a="rna", mod_b="atac",
    var_names_by_mod={"rna": test["rna"].var_names, "atac": test["atac"].var_names},
    tokenizer_mode_by_mod={"rna": "topk_channels", "atac": "topk_channels"},
    layer=-1, top_pairs_per_cell=50, top_n=25)
top_pairs = pd.DataFrame(pairs, columns=["gene", "peak", "summed_attention"])
if tss is not None and peak_chrom is not None:
    pidx = test["atac"].var_names.get_indexer(top_pairs["peak"])
    g = tss.reindex(top_pairs["gene"])
    same = g["chrom"].to_numpy() == peak_chrom[pidx]
    top_pairs["same_chromosome"] = same
    top_pairs["distance_to_tss_kb"] = np.where(same, np.abs(peak_mid[pidx] - g["tss"].to_numpy()) / 1e3, np.nan)
top_pairs.round(3)

### Do gene tokens attend more to their cis peaks?

In every cell, a gene token (query) attends to all peak tokens (keys). A (gene, peak) pair is **cis** if
the peak lies on the gene's chromosome within `CIS_WINDOW_BP` of its TSS. A naive test ("are cis pairs
over-represented among the highest attention weights?") is confounded: some peaks receive high
attention from every gene and some genes send much of their attention to ATAC, and if such tokens are
more or less often cis, the naive test reports enrichment without any pair-specific preference. (On the
synthetic stand-in data used for our automated tests, which has no gene-peak structure at all, the
naive version reported a spurious 2-3x enrichment.)

The test used here removes those effects first. For each cell, the last layer's gene → peak attention
block is **double-centered** (each gene's mean and each peak's mean are subtracted). Then, for every
gene token that has at least one cis and one non-cis peak token, we compute the **AUC** with which its
centered attention separates cis from non-cis peaks (0.5 = no preference) and average over gene tokens.

The same gene–peak pairs recur in many cells, so cells are not independent samples and a bootstrap over
cells would be too optimistic. Instead the statistic is calibrated with a **permutation null**: peak
coordinates are shuffled across peaks (`N_PERMUTATIONS` times), which keeps the attention exactly as it
is but breaks its relation to genomic position, and the mean AUC is recomputed.

In [ ]:
def mean_cis_auc(R, gi, pi, chrom_of_peak, mid_of_peak):
    """Mean within-gene AUC (cis vs non-cis peak tokens) of centered attention R (cells, genes, peaks)."""
    cis = ((g_chrom[gi][:, :, None] == chrom_of_peak[pi][:, None, :]) &
           (np.abs(g_tss[gi][:, :, None] - mid_of_peak[pi][:, None, :]) <= CIS_WINDOW_BP) &
           gene_ok[gi][:, :, None])
    n1 = cis.sum(2)
    n0 = cis.shape[2] - n1
    ok = (n1 > 0) & (n0 > 0)
    ranks = R.argsort(2).argsort(2) + 1.0                                  # within-row ranks (ties negligible)
    u = (ranks * cis).sum(2) - n1 * (n1 + 1) / 2
    auc = np.where(ok, u / np.maximum(n1 * n0, 1), np.nan)
    return np.nanmean(auc) if ok.any() else np.nan, int(ok.sum())


if tss is None or peak_chrom is None:
    print("Gene or peak coordinates are missing; skipping the cis test.")
else:
    gene_ok = test["rna"].var_names.isin(tss.index)
    g_chrom = np.full(test["rna"].n_vars, "", dtype=object)
    g_tss = np.zeros(test["rna"].n_vars, dtype=np.int64)
    g_chrom[gene_ok] = tss.loc[test["rna"].var_names[gene_ok], "chrom"].to_numpy()
    g_tss[gene_ok] = tss.loc[test["rna"].var_names[gene_ok], "tss"].to_numpy()
    Rs, GIs, PIs = [], [], []
    for s, tokmap, attn in attention_batches():
        shift = 1 if tokmap.has_global_cls else 0
        a0, a1 = [v + shift for v in tokmap.slices["rna"]]
        b0, b1 = [v + shift for v in tokmap.slices["atac"]]
        A = attn[-1][:, a0:a1, b0:b1].float().cpu().numpy()                 # (cells, gene tokens, peak tokens)
        Rs.append(A - A.mean(2, keepdims=True) - A.mean(1, keepdims=True) + A.mean((1, 2), keepdims=True))
        GIs.append(tokmap.meta["rna"]["topk_idx"].cpu().numpy())            # feature index of each token
        PIs.append(tokmap.meta["atac"]["topk_idx"].cpu().numpy())
    R, GI, PI = np.concatenate(Rs), np.concatenate(GIs), np.concatenate(PIs)
    observed_auc, n_rows = mean_cis_auc(R, GI, PI, peak_chrom, peak_mid)
    rng = np.random.default_rng(0)
    null_auc = []
    for _ in range(N_PERMUTATIONS):
        perm = rng.permutation(len(peak_chrom))
        null_auc.append(mean_cis_auc(R, GI, PI, peak_chrom[perm], peak_mid[perm])[0])
    null_auc = np.asarray(null_auc)
    null_auc = null_auc[np.isfinite(null_auc)]
    if not np.isfinite(observed_auc):
        print("No gene token had both cis and non-cis peak tokens; widen CIS_WINDOW_BP or raise the token counts.")
    else:
        p_perm = (1 + (null_auc >= observed_auc).sum()) / (1 + len(null_auc))
        print(f"eligible gene tokens: {n_rows:,} in {len(R):,} cells")
        print(f"mean within-gene AUC (cis vs non-cis peaks): {observed_auc:.3f}; permutation null "
              f"{null_auc.mean():.3f} ± {null_auc.std():.3f}; one-sided p = {p_perm:.3f} ({len(null_auc)} permutations)")
        fig, ax = plt.subplots(figsize=(4, 2.6))
        ax.hist(null_auc, bins=25, color="0.7", label="shuffled peak coordinates")
        ax.axvline(observed_auc, c="C3", label="observed")
        ax.set(xlabel="mean within-gene AUC", ylabel="permutations")
        ax.legend(frameon=False, fontsize=7)
        plt.show()

An observed value inside the permutation distribution means the last layer shows no preference for cis
peaks at this window. A value clearly above it would suggest the fused encoder links genes to nearby regulatory
regions, and would be worth testing against known enhancer–gene maps and across seeds. Earlier layers
can be tested by changing `attn[-1]`.

## 5. Saving and restoring a transformer model

Save the weights and rebuild the model from the same configuration code; `strict=True` checks that
every parameter (including the feature-ID embeddings) is restored.

In [ ]:
torch.save(fused.state_dict(), "fused_transformer_state.pt")
cfg, kwargs = specs["fused transformer (v2)"]
restored = UniVIMultiModalVAE(cfg, normalize_v1_terms=True, **kwargs).to(device)
restored.load_state_dict(torch.load("fused_transformer_state.pt", map_location=device, weights_only=True), strict=True)
same = np.allclose(encode_fused_adata_pair(restored, {"rna": test["rna"], "atac": test["atac"]}, device=device,
                                           write_to_adatas=False)["mu"], fused_z["fused transformer (v2)"], atol=1e-5)
print("restored model reproduces the fused embedding:", same)

## Notes and limitations

- **Objective vs. architecture.** Compare "fused transformer (v2)" with "MLP (v2)", and "transformer
  (v1)" with "MLP (v1)"; comparing across objectives mixes two changes.
- **Tokens drop information.** Each cell contributes only its top `N_TOKENS_*` features to a transformer
  encoder; the MLP sees every feature. Larger token counts cost memory quadratically in the fused model.
- **Saving.** With the tokenizer subclass, save the `state_dict` and rebuild the model from this
  notebook's configuration code (Section 5); `load_reference` cannot rebuild the subclass.
- **Genomic coordinates.** The tokenizer has an option for peak-coordinate embeddings. In UniVI 1.1 the
  coordinate MLP normalizes a single scalar with `LayerNorm`, which makes its output independent of the
  position (only the chromosome embedding carries information), so it is not used here.